# EDA — Phase-transition video curation

Lance-native exploration of `videos_raw` for the videogen pipeline.

Sections:
1. Tier-1 summary — caption keyword counts.
2. Source distributions (duration, fps, caption length).
3. Per-transition rates + samples.
4. Full-text search.
5. **Tier-2** — CLIP text→video retrieval, motion + MTScore filtering.
6. **Tier-2** quality-gated `phase_transitions_curated_*` view.
7. **Tier-4** dedup stats.

Prereqs (run from the example root):

```
python -m videogen.ingest_chronomagic --synthetic 500 --overwrite
python -m videogen.backfill_geneva --tier 1
python -m videogen.backfill_geneva --tier 2
python -m videogen.manage_views --action curate
python -m videogen.manage_views --action curate-2
```

In [ ]:
import lancedb, pyarrow as pa, pandas as pd, numpy as np
from videogen.spec_queries import (
    summarise, distribution, preview, fts, ensure_caption_fts,
    keyword_spec, union_keyword_spec, curated_spec,
)
from videogen.schema import PHASE_TRANSITIONS

DB = 'data/videos/lancedb'
tbl = lancedb.connect(DB).open_table('videos_raw')
print(f'rows={len(tbl):,}  version={tbl.version}')

## 1 · Tier-1 summary

In [ ]:
pd.DataFrame(summarise(tbl), columns=['slice', 'rows'])

## 2 · Distributions

In [ ]:
df = tbl.search().select(['duration_s', 'fps', 'n_frames', 'caption_length']).to_pandas()
df.describe()

In [ ]:
ax = df['caption_length'].hist(bins=40)
ax.set_title('caption length distribution')
ax.set_xlabel('characters')
ax.set_ylabel('clips')

## 3 · Per-transition rates

In [ ]:
rows = []
for t in PHASE_TRANSITIONS:
    rows.append((t,
                  tbl.count_rows(filter=keyword_spec(t, 'train')),
                  tbl.count_rows(filter=keyword_spec(t, 'val'))))
pd.DataFrame(rows, columns=['transition', 'train', 'val'])

In [ ]:
preview(tbl, keyword_spec('melting'), n=5, columns=['clip_id', 'caption', 'duration_s'])

## 4 · FTS — surface clips a keyword regex misses

In [ ]:
ensure_caption_fts(tbl)
fts(tbl, 'ice cube', n=5)

## 5 · Tier-2 — CLIP text → video retrieval

Pre-computed `clip_emb_video` lets us search for clips with a natural-
language query, without ever touching the actual video bytes again.
Build the vector index once, query as many times as you like.

In [ ]:
# Build cosine IVF index on clip_emb_video (idempotent — re-running is cheap)
try:
    tbl.create_index(metric='cosine', vector_column_name='clip_emb_video',
                     index_type='IVF_FLAT',
                     num_partitions=max(2, len(tbl) // 64))
    print('clip_emb_video index ready')
except Exception as e:
    print('index already exists or:', e)

In [ ]:
import open_clip, torch
m, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai',
                                                 device='cuda')
tok = open_clip.get_tokenizer('ViT-B-32')
with torch.no_grad():
    q = m.encode_text(tok(['ice melting into water']).cuda())
    q = (q / q.norm(dim=-1, keepdim=True)).cpu().float()[0].tolist()

tbl.search(q, vector_column_name='clip_emb_video') \
   .metric('cosine').limit(5) \
   .to_pandas()[['clip_id', 'caption', '_distance']]

## 6 · Curated (Tier-2 quality gate)

`phase_transitions_curated_train` filters the keyword union further by
motion + MTScore + duration.  This is the view the training script reads.

In [ ]:
print('curated count:', tbl.count_rows(filter=curated_spec('train')))
tbl.search().where(curated_spec('train')) \
   .select(['clip_id', 'caption', 'motion_strength', 'metamorphic_score']) \
   .limit(5).to_pandas()

In [ ]:
# Motion + MTScore joint distribution — visualize how the gate carves up the corpus
df = tbl.search().select(['motion_strength', 'metamorphic_score']).to_pandas()
ax = df.plot.scatter('motion_strength', 'metamorphic_score', alpha=0.5)
ax.axvspan(2.0, 12.0, alpha=0.05, color='green')
ax.axhline(0.6, linestyle='--', color='gray')
ax.set_title('Tier-2 quality gate: motion ∈ [2, 12]  ∩  MTScore > 0.6')

## 7 · Tier-4 dedup

Once `dhash_first_last` is backfilled and the L2 index is built,
`is_duplicate` flags clips whose first+last fingerprint is within
Hamming distance ≤ 12 of another clip.

Each MV created with `manage_views --action curate` automatically
applies `is_duplicate = false` on its `_train` slice (see
`manage_views._dedup_clause`).

In [ ]:
if 'is_duplicate' in tbl.schema.names:
    n_dup = tbl.count_rows(filter='is_duplicate = true')
    n_keep = tbl.count_rows(filter='is_duplicate IS NULL OR is_duplicate = false')
    print(f'duplicates: {n_dup}/{len(tbl)} ({n_dup/len(tbl)*100:.1f}%) — training-eligible: {n_keep}')
else:
    print('Tier-4 not run yet — see KNOWN_ISSUES.md for the dedup workflow')